# AgriTrust — Risk Scorer Training
Trains a Random Forest classifier to identify high-risk users based on transaction behaviour.

**Output:** `ml_weights/risk_scorer_v4.pkl`

In [ ]:
import sys, os
# Point to repo root so backend imports work
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, 'data/training'))
print('Repo root:', REPO_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data
Loads real user transaction data from the database. Falls back to synthetic data if DB is unavailable.

In [ ]:
def load_data():
    try:
        from backend.app.db.session import SessionLocal
        from backend.app.models.user import User
        from backend.app.models.transaction import Transaction
        db = SessionLocal()
        users = db.query(User).all()
        if len(users) < 10:
            raise ValueError('Not enough users in DB yet')
        rows = []
        for u in users:
            total = u.total_transactions or 0
            rows.append({
                'total_transactions': total,
                'successful_transactions': u.successful_transactions or 0,
                'disputed_transactions': u.disputed_transactions or 0,
                'avg_transaction_value': float(u.avg_transaction_value or 0),
                'account_age_days': (pd.Timestamp.utcnow() - pd.Timestamp(u.created_at)).days if u.created_at else 0,
                'is_verified': 1 if u.is_verified else 0,
                'trust_score': float(u.trust_score or 50),
                'region': u.region or 'Unknown',
                'dispute_rate': (u.disputed_transactions or 0) / (total + 1),
                'is_high_risk': 1 if (u.risk_score or 0) >= 60 else 0,
            })
        db.close()
        df = pd.DataFrame(rows)
        print(f'Loaded {len(df)} users from database')
        return df
    except Exception as e:
        print(f'DB unavailable ({e}) — using synthetic data')
        from utils.data_loader import generate_sample_data
        return generate_sample_data('user_transactions', n_samples=10000)

df = load_data()
print(df.shape)
df.head()

## 2. Feature Engineering

In [ ]:
features = pd.DataFrame()
features['txn_count']          = df['total_transactions']
features['success_rate']       = df['successful_transactions'] / (df['total_transactions'] + 1)
features['dispute_rate']       = df['dispute_rate']
features['avg_txn_value']      = df['avg_transaction_value']
features['account_age_months'] = df['account_age_days'] / 30
features['is_new_user']        = (df['account_age_days'] < 30).astype(int)
features['is_verified']        = df['is_verified']
features['trust_score']        = df['trust_score']
features['risk_interaction']   = features['dispute_rate'] * (1 - features['success_rate'])

region_dummies = pd.get_dummies(df['region'], prefix='region')
features = pd.concat([features, region_dummies], axis=1)
target = df['is_high_risk']

print(f'Features: {list(features.columns)}')
print(f'High-risk rate: {target.mean()*100:.1f}%')

## 3. Train / Test Split + Scale

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f'Train: {len(X_train)}  Test: {len(X_test)}')

## 4. Train Model

In [ ]:
model = RandomForestClassifier(
    n_estimators=100, max_depth=10,
    min_samples_split=5, random_state=42,
    class_weight='balanced', n_jobs=-1
)
model.fit(X_train_s, y_train)
print('Training complete')

## 5. Evaluate

In [ ]:
y_pred  = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]
print(classification_report(y_test, y_pred))
print(f'AUC: {roc_auc_score(y_test, y_proba):.4f}')

cv = cross_val_score(model, X_train_s, y_train, cv=5)
print(f'CV: {cv.mean():.4f} ± {cv.std()*2:.4f}')

## 6. Feature Importance

In [ ]:
imp = pd.Series(model.feature_importances_, index=features.columns).sort_values(ascending=False)
imp.head(10).plot(kind='barh', figsize=(8, 5), title='Top 10 Feature Importances')
plt.tight_layout()
plt.show()

## 7. Save

In [ ]:
WEIGHTS = os.path.join(REPO_ROOT, 'ml_weights')
os.makedirs(WEIGHTS, exist_ok=True)
joblib.dump(model,                    f'{WEIGHTS}/risk_scorer_v4.pkl')
joblib.dump(scaler,                   f'{WEIGHTS}/risk_scaler_v4.pkl')
joblib.dump(features.columns.tolist(),f'{WEIGHTS}/risk_features_v4.pkl')
print('Saved to ml_weights/')